In [1]:
import joblib

joblib.cpu_count(only_physical_cores=False)

10

In [2]:
%%writefile bench_script.py
import os
import json
import sys
import joblib
import threadpoolctl
import joblib
from time import perf_counter
from sklearn.metrics import log_loss

params = json.loads(sys.argv[1])


module = __import__(params["module"], fromlist=[params["estimator"]])
estimator_class = getattr(module, params["estimator"])
X, y = joblib.load(params["data_filepath"])

if "threadpool_limits" in params:
    threadpoolctl.threadpool_limits(limits=params["threadpool_limits"])

est = estimator_class()

tic = perf_counter()
count = 0
while perf_counter() - tic < 3.0:
    count += 1
    est.fit(X, y)
toc = perf_counter()

duration = (toc - tic) / count

log_loss_value = log_loss(y, est.predict_proba(X))


results = {
    "cpu_count": joblib.cpu_count(),
    "cpu_count_physical": joblib.cpu_count(only_physical_cores=True),
    "threadpool_info": threadpoolctl.threadpool_info(),
    "fit_time_seconds": duration,
    "n_iterations": count,
    "n_samples": X.shape[0],
    "n_features": X.shape[1],
    "n_classes": est.classes_.shape[0],
    "log_loss": log_loss_value,
}
with open(f"results_{params.get('run_id', 'default')}.json", "w") as f:
    json.dump(results, f)


Writing bench_script.py


In [4]:
import os
import subprocess
import sys
import json
import joblib
from pprint import pprint
from sklearn.datasets import make_classification


X, y = make_classification(n_samples=int(1e2), n_features=20, random_state=0)
data_filename = f"data_{joblib.hash([X, y])}.pkl"
joblib.dump((X, y), data_filename)


params = {
    "data_filepath": data_filename,
    "threadpool_limits": 1,
    "module": "sklearn.ensemble",
    "estimator": "HistGradientBoostingClassifier",
    # "module": "lightgbm",
    # "estimator": "LGBMClassifier",
    # "module": "xgboost",
    # "estimator": "XGBClassifier",
}

run_id = joblib.hash(params)
params.update(
    {
        "run_id": run_id,
    }
)

env = os.environ.copy()
env.update(
    {
        # "OMP_NUM_THREADS": "1",
    }
)
out = subprocess.run(
    [sys.executable, "bench_script.py", json.dumps(params)],
    capture_output=True,
    text=True,
    env=env,
)


if out.returncode != 0:
    print(out.stderr)
    print(out.stdout)
else:
    with open(f"results_{run_id}.json", "r") as f:
        results = json.load(f)

    results["stdout"] = out.stdout
    results["stderr"] = out.stderr
    pprint(results)

{'cpu_count': 10,
 'cpu_count_physical': 10,
 'fit_time_seconds': 0.013994990698560033,
 'log_loss': 0.02022472899126647,
 'n_classes': 2,
 'n_features': 20,
 'n_iterations': 215,
 'n_samples': 100,
 'stderr': '',
 'stdout': '',
 'threadpool_info': [{'architecture': 'VORTEX',
                      'filepath': '/Users/ogrisel/miniforge3/envs/dev/lib/libopenblas.0.dylib',
                      'internal_api': 'openblas',
                      'num_threads': 1,
                      'prefix': 'libopenblas',
                      'threading_layer': 'openmp',
                      'user_api': 'blas',
                      'version': '0.3.30'},
                     {'filepath': '/Users/ogrisel/miniforge3/envs/dev/lib/libomp.dylib',
                      'internal_api': 'openmp',
                      'num_threads': 1,
                      'prefix': 'libomp',
                      'user_api': 'openmp',
                      'version': None}]}
